In [3]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet

import xgboost as xgb
from lightgbm import LGBMRegressor

import joblib 

## Fields + Wether 

In [4]:
all_yields_df_60_weather = pd.read_csv("./data_1/all_yields_60_.csv")
all_yields_df_62_weather = pd.read_csv("./data_1/all_yields_.csv")

In [5]:
yield_df = pd.concat([all_yields_df_60_weather, all_yields_df_62_weather], axis=0)

In [6]:
len(all_yields_df_60_weather) + len(all_yields_df_62_weather)

552264

In [7]:
len(yield_df)

552264

In [9]:
# yield_df.columns

In [10]:
yield_df['Organic_M'] = yield_df['Organic M.'].combine_first(yield_df['Organic M'])

C:\Users\katja\AppData\Local\Temp\ipykernel_8048\2586215567.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  yield_df['Organic_M'] = yield_df['Organic M.'].combine_first(yield_df['Organic M'])


In [135]:
# yield_df["Crop_174"]

In [11]:
yield_zones = yield_df.groupby(['area', 'year']).mean().reset_index()

In [12]:
yield_zones_ = yield_zones.drop(columns=["SECTIONID", 'Organic M.', 'Organic M', 
                                         'xcoord', 'ycoord'])

In [10]:
# yield_zones_

In [13]:
yield_zones_["Crop_22"].isna().sum()

147

In [14]:
yield_zones_['Crop_22'] = yield_zones_['Crop_22'].fillna(False).astype(bool)
yield_zones_['Crop_33'] = yield_zones_["Crop_33"].fillna(False).astype(bool)
yield_zones_['Crop_34'] = yield_zones_['Crop_34'].astype(bool)
yield_zones_['Crop_41'] = yield_zones_['Crop_41'].astype(bool)
yield_zones_['Crop_45'] = yield_zones_['Crop_45'].astype(bool)
yield_zones_['Crop_174'] = yield_zones_['Crop_174'].astype(bool)

In [15]:
yield_zones_["Crop_22"].isna().sum()

0

In [16]:
yield_zones_.isna().sum()

area                       0
year                       0
VRYIELDMAS                 0
Moisture                   0
Elevation                  0
                          ..
std_cloudcover_month_8     0
std_cloudcover_month_9     0
std_cloudcover_month_10    0
Crop_22                    0
Organic_M                  0
Length: 170, dtype: int64

In [17]:
len(yield_zones_)

228

In [14]:
feature_names = ["Elevation", "area", "Organic_M", "Ca", "Mg", "Mn", "B",
                 "Cu", "Mo", "Fe", "Zn", "S", "P", "K", "Na", "pH", "C.E.C",
                 "year", "Crop_22", "Crop_34", "Crop_41", "Crop_45", "Crop_174", 

    "mean_tempmax_month_5", "mean_tempmax_month_6", "mean_tempmax_month_7",
    "mean_tempmax_month_8", "mean_tempmax_month_9", "mean_tempmax_month_10",
    "std_tempmax_month_5", "std_tempmax_month_6", "std_tempmax_month_7",
    "std_tempmax_month_8", "std_tempmax_month_9", "std_tempmax_month_10",

    "mean_tempmin_month_5", "mean_tempmin_month_6", "mean_tempmin_month_7",
    "mean_tempmin_month_8", "mean_tempmin_month_9", "mean_tempmin_month_10",
    "std_tempmin_month_5", "std_tempmin_month_6", "std_tempmin_month_7",
    "std_tempmin_month_8", "std_tempmin_month_9", "std_tempmin_month_10",

    "mean_temp_month_5", "mean_temp_month_6", "mean_temp_month_7",
    "mean_temp_month_8", "mean_temp_month_9", "mean_temp_month_10",
    "std_temp_month_5", "std_temp_month_6", "std_temp_month_7",
    "std_temp_month_8", "std_temp_month_9", "std_temp_month_10",
    
    "mean_humidity_month_5", "mean_humidity_month_6", "mean_humidity_month_7",
    "mean_humidity_month_8", "mean_humidity_month_9", "mean_humidity_month_10",
    "std_humidity_month_5", "std_humidity_month_6", "std_humidity_month_7",
    "std_humidity_month_8", "std_humidity_month_9", "std_humidity_month_10",
    
    "mean_precip_month_5", "mean_precip_month_6", "mean_precip_month_7",
    "mean_precip_month_8", "mean_precip_month_9", "mean_precip_month_10",
    "std_precip_month_5", "std_precip_month_6", "std_precip_month_7",
    "std_precip_month_8", "std_precip_month_9", "std_precip_month_10",
    
    "mean_cloudcover_month_5", "mean_cloudcover_month_6", "mean_cloudcover_month_7",
    "mean_cloudcover_month_8", "mean_cloudcover_month_9", "mean_cloudcover_month_10",
    "std_cloudcover_month_5", "std_cloudcover_month_6", "std_cloudcover_month_7",
    "std_cloudcover_month_8", "std_cloudcover_month_9", "std_cloudcover_month_10",
    ]

### Regressions

#### MLR

In [15]:
X_mlr_ = yield_zones_[feature_names]
y_mlr_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_mlr_ = scaler.fit_transform(X_mlr_)

X_train_mlr_, X_test_mlr_, y_train_mlr_, y_test_mlr_ = train_test_split(X_mlr_, y_mlr_, test_size=0.2, random_state=0)

In [16]:
print(len(X_train_mlr_))
print(len(X_test_mlr_))
print(len(y_train_mlr_))
print(len(y_test_mlr_))

182
46
182
46


In [17]:
reg_model_mlr_ = LinearRegression(copy_X=True, fit_intercept=True, n_jobs=15, positive=True)
reg_model_mlr_ = LinearRegression().fit(X_train_mlr_, y_train_mlr_)
print('Intercept: ', reg_model_mlr_.intercept_)
y_pred_mlr_ = reg_model_mlr_.predict(X_test_mlr_)  
print("Prediction for test set: {}".format(y_pred_mlr_))

Intercept:  5.368614576845946
Prediction for test set: [ 0.78595729  3.54330685  1.56515309 14.88927799 10.3012195  10.5951822
  1.74218358  0.48343766 11.11544847 11.04343184 11.11027494  3.07228439
  2.95422046  1.32624034  3.13726729  2.55934031  1.37691953 14.11265101
  3.22722572 12.61677318  3.2044072   1.76376958  3.37831368  0.98402756
  3.4926936   1.22751251 13.36549001  2.59794834 14.61035577  1.59465844
  2.82222881  2.94105458  3.62510254  2.41976641 14.38372177  2.55921039
  1.24331507  1.60587082  3.08342359  2.86229512 15.06710005  2.82783352
  3.37725324  3.35178774  3.29076646  3.51094198]


In [18]:
mae_mlr_ = metrics.mean_absolute_error(y_test_mlr_, y_pred_mlr_)
mse_mlr_ = metrics.mean_squared_error(y_test_mlr_, y_pred_mlr_)
rmse_mlr_ = np.sqrt(metrics.mean_squared_error(y_test_mlr_, y_pred_mlr_))
r_2_mlr_ = r2_score(y_test_mlr_, y_pred_mlr_)

print('Mean Absolute Error:', mae_mlr_)
print('Mean Square Error:', mse_mlr_)
print('Root Mean Square Error:', rmse_mlr_)
print('R^2:', r_2_mlr_)

Mean Absolute Error: 0.44840039704096823
Mean Square Error: 0.39379584634781156
Root Mean Square Error: 0.6275315500815968
R^2: 0.9815469629398482


In [268]:
# param_grid = {
#     'copy_X': [True,False], 
#     'fit_intercept': [True,False], 
#     'n_jobs': [1, 5, 10, 15, None], 
#     'positive': [True,False]
#     }

# halving_grid_search_mlr = HalvingGridSearchCV(
#     reg_model_mlr_, 
#     param_grid, 
#     scoring='r2',
#     cv=5, 
#     n_jobs=-1,
#     )
# halving_grid_search_mlr.fit(X_train_mlr_, y_train_mlr_)

# print(f"Best Hyperparameters: {halving_grid_search_mlr.best_params_}")
# print(f"Best Score: {halving_grid_search_mlr.best_score_}")

Best Hyperparameters: {'copy_X': True, 'fit_intercept': True, 'n_jobs': 15, 'positive': True}

#### Lasso

In [19]:
X_lasso_ = yield_zones_[feature_names]
y_lasso_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_lasso_ = scaler.fit_transform(X_lasso_)

X_train_lasso_, X_test_lasso_, y_train_lasso_, y_test_lasso_ = train_test_split(X_lasso_, y_lasso_, test_size=0.2, random_state=0)

In [20]:
lasso_model_ = Lasso(alpha=0.1)
lasso_model_.fit(X_train_lasso_, y_train_lasso_)

Lasso(alpha=0.1)

In [21]:
y_pred_lasso_ = lasso_model_.predict(X_test_lasso_)

mae_lasso_ = metrics.mean_absolute_error(y_test_lasso_, y_pred_lasso_)
mse_lasso_ = metrics.mean_squared_error(y_test_lasso_, y_pred_lasso_)
rmse_lasso_ = np.sqrt(metrics.mean_squared_error(y_test_lasso_, y_pred_lasso_))
r_2_lasso_ = r2_score(y_test_lasso_, y_pred_lasso_)

print('Mean Absolute Error:', mae_lasso_)
print('Mean Square Error:', mse_lasso_)
print('Root Mean Square Error:', rmse_lasso_)
print('R^2:', r_2_lasso_)

Mean Absolute Error: 0.4410205020509203
Mean Square Error: 0.4157097680112476
Root Mean Square Error: 0.644755587809247
R^2: 0.9805200897202879


In [253]:
param_grid_lasso = {
    'alpha' : [0.1, 1, 5, 10, 20, 50, 100]
    }

grid_search_lasso = HalvingGridSearchCV(
    lasso_model_, 
    param_grid_lasso, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
    )
grid_search_lasso.fit(X_train_lasso_, y_train_lasso_)

print(f"Best Hyperparameters: {grid_search_lasso.best_params_}")
print(f"Best Score: {grid_search_lasso.best_score_}")

Best Hyperparameters: {'alpha': 0.1}
Best Score: 0.9867666751572658


#### Ridge

In [22]:
X_ridge_ = yield_zones_[feature_names]
y_ridge_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_ridge_ = scaler.fit_transform(X_ridge_)

X_train_ridge_, X_test_ridge_, y_train_ridge_, y_test_ridge_ = train_test_split(X_ridge_, y_ridge_, test_size=0.2, random_state=0)

In [23]:
ridge_model_ = Ridge(alpha=5)
ridge_model_.fit(X_train_ridge_, y_train_ridge_)

Ridge(alpha=5)

In [24]:
y_pred_ridge_ = ridge_model_.predict(X_test_ridge_)

mae_ridge_ = metrics.mean_absolute_error(y_test_ridge_, y_pred_ridge_)
mse_ridge_ = metrics.mean_squared_error(y_test_ridge_, y_pred_ridge_)
rmse_ridge_ = np.sqrt(metrics.mean_squared_error(y_test_ridge_, y_pred_ridge_))
r_2_ridge_ = r2_score(y_test_ridge_, y_pred_ridge_)

print('Mean Absolute Error:', mae_ridge_)
print('Mean Square Error:', mse_ridge_)
print('Root Mean Square Error:', rmse_ridge_)
print('R^2:', r_2_ridge_)

Mean Absolute Error: 0.4330118171117772
Mean Square Error: 0.38370620477209
Root Mean Square Error: 0.6194402350284408
R^2: 0.9820197574897328


In [275]:
param_grid_ridge = {
    'alpha' : [0.0001, 0.001, 0.01, 0.1, 1, 5, 10, 20, 50, 100]
    }

grid_search_ridge = HalvingGridSearchCV(
    ridge_model_, 
    param_grid_ridge, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
    )
grid_search_ridge.fit(X_train_ridge_, y_train_ridge_)

print(f"Best Hyperparameters: {grid_search_ridge.best_params_}")
print(f"Best Score: {grid_search_ridge.best_score_}")

Best Hyperparameters: {'alpha': 5}
Best Score: 0.9879314760358883


#### Polynomial

In [25]:
X_pol_ = yield_zones_[feature_names]
y_pol_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_pol_ = scaler.fit_transform(X_pol_)

X_train_pol_, X_test_pol_, y_train_pol_, y_test_pol_ = train_test_split(X_pol_, y_pol_, test_size=0.2, random_state=0)

In [26]:
lin_pol_ = LinearRegression()
lin_pol_.fit(X_train_pol_, y_train_pol_)

poly_ = PolynomialFeatures(degree=3)
X_poly_ = poly_.fit_transform(X_train_pol_)
X_poly_test_ = poly_.fit_transform(X_test_pol_)

poly_.fit(X_poly_, y_train_pol_)
lin2_pol_ = LinearRegression()
lin2_pol_.fit(X_poly_, y_train_pol_)

LinearRegression()

In [27]:
y_pred_pol_ = lin2_pol_.predict(X_poly_test_)

mae_pol_ = metrics.mean_absolute_error(y_test_pol_, y_pred_pol_)
mse_pol_ = metrics.mean_squared_error(y_test_pol_, y_pred_pol_)
rmse_pol_ = np.sqrt(metrics.mean_squared_error(y_test_pol_, y_pred_pol_))
r_2_pol_ = r2_score(y_test_pol_, y_pred_pol_)

print('Mean Absolute Error:', mae_pol_)
print('Mean Square Error:', mse_pol_)
print('Root Mean Square Error:', rmse_pol_)
print('R^2:', r_2_pol_)

Mean Absolute Error: 0.8506582810413575
Mean Square Error: 1.1485536264631135
Root Mean Square Error: 1.0717059421609612
R^2: 0.9461794662608599


In [28]:
mae_ = []
mse_ = []
rmse_ = []
r2_ = []

degrees = [2, 3, 4]

for degree in degrees:
    pipeline = Pipeline([('poly_features', PolynomialFeatures(degree=degree)), ('model', LinearRegression())])
    pipeline.fit(X_train_pol_, y_train_pol_)

    y_pred_test_ = pipeline.predict(X_test_pol_)

    mae_.append(metrics.mean_absolute_error(y_test_pol_, y_pred_pol_))
    mse_.append(metrics.mean_squared_error(y_test_pol_, y_pred_pol_))
    rmse_.append(np.sqrt(metrics.mean_squared_error(y_test_pol_, y_pred_pol_)))
    r2_.append(r2_score(y_test_pol_, y_pred_test_))


In [29]:
print(degrees,'\n')
print("MAE: ", mae_,'\n')
print("MSE: ", mse_, "\n")
print("RMSE: ", rmse_, "\n")
print("R2: ", r2_, "\n")

[2, 3, 4] 

MAE:  [0.8506582810413575, 0.8506582810413575, 0.8506582810413575] 

MSE:  [1.1485536264631135, 1.1485536264631135, 1.1485536264631135] 

RMSE:  [1.0717059421609612, 1.0717059421609612, 1.0717059421609612] 

R2:  [-1884.2709898484636, 0.9461794662608599, 0.9632032956603582] 



#### Decision Tree

In [30]:
X_dt_ = yield_zones_[feature_names]
y_dt_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_dt_ = scaler.fit_transform(X_dt_)

X_train_dt_, X_test_dt_, y_train_dt_, y_test_dt_ = train_test_split(X_dt_, y_dt_, test_size=0.2, random_state=0)

In [31]:
regressor_dt_ = DecisionTreeRegressor(max_depth=6, max_leaf_nodes=64, random_state=42)
regressor_dt_.fit(X_train_dt_, y_train_dt_)

DecisionTreeRegressor(max_depth=6, max_leaf_nodes=64, random_state=42)

In [32]:
y_pred_dt_ = regressor_dt_.predict(X_test_dt_)

mae_dt_ = metrics.mean_absolute_error(y_test_dt_, y_pred_dt_)
mse_dt_ = metrics.mean_squared_error(y_test_dt_, y_pred_dt_)
rmse_dt_ = np.sqrt(metrics.mean_squared_error(y_test_dt_, y_pred_dt_))
r_2_dt_ = r2_score(y_test_dt_, y_pred_dt_)

print('Mean Absolute Error:', mae_dt_)
print('Mean Square Error:', mse_dt_)
print('Root Mean Square Error:', rmse_dt_)
print('R^2:', r_2_dt_)

Mean Absolute Error: 0.5125931330086475
Mean Square Error: 0.5324508807693933
Root Mean Square Error: 0.7296923192479097
R^2: 0.9750496712276897


In [286]:
param_grid_dt = {
    'max_depth' : [4, 6, 8, 10], 
    'max_leaf_nodes' : [16, 32, 64]
}

halving_grid_search_dt = HalvingGridSearchCV(
    regressor_dt_, 
    param_grid_dt, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_dt.fit(X_train_dt_, y_train_dt_)


print(f"Best Hyperparameters: {halving_grid_search_dt.best_params_}")
print(f"Best Score: {halving_grid_search_dt.best_score_}")

Best Hyperparameters: {'max_depth': 6, 'max_leaf_nodes': 64}
Best Score: 0.983555504186038


#### Random Forest

In [33]:
X_rf_ = yield_zones_[feature_names]
y_rf_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_rf_ = scaler.fit_transform(X_rf_)

X_train_rf_, X_test_rf_, y_train_rf_, y_test_rf_ = train_test_split(X_rf_, y_rf_, test_size=0.2, random_state=0)

In [34]:
regr_rf_ = RandomForestRegressor(max_depth=10, max_leaf_nodes=32, n_estimators=100)
regr_rf_.fit(X_train_rf_, y_train_rf_)

RandomForestRegressor(max_depth=10, max_leaf_nodes=32)

In [35]:
y_pred_rf_ = regr_rf_.predict(X_test_rf_)

mae_rf_ = metrics.mean_absolute_error(y_test_rf_, y_pred_rf_)
mse_rf_ = metrics.mean_squared_error(y_test_rf_, y_pred_rf_)
rmse_rf_ = np.sqrt(metrics.mean_squared_error(y_test_rf_, y_pred_rf_))
r_2_rf_ = r2_score(y_test_rf_, y_pred_rf_)

print('Mean Absolute Error:', mae_rf_)
print('Mean Square Error:', mse_rf_)
print('Root Mean Square Error:', rmse_rf_)
print('R^2:', r_2_rf_)

Mean Absolute Error: 0.4246542779548671
Mean Square Error: 0.463815061796555
Root Mean Square Error: 0.681039691792303
R^2: 0.9782659045193964


In [292]:
param_grid_rf = {
    'max_depth' : [6, 8, 10], 
    'max_leaf_nodes' : [32, 64], 
    'n_estimators' : [100, 200, 300]
}

halving_grid_search_rf = HalvingGridSearchCV(
    regr_rf_, 
    param_grid_rf, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_rf.fit(X_train_rf_, y_train_rf_)


print(f"Best Hyperparameters: {halving_grid_search_rf.best_params_}")
print(f"Best Score: {halving_grid_search_rf.best_score_}")

Best Hyperparameters: {'max_depth': 10, 'max_leaf_nodes': 32, 'n_estimators': 100}
Best Score: 0.9907182735543051


#### KNN

In [36]:
X_knn_ = yield_zones_[feature_names]
y_knn_ = yield_zones_["VRYIELDMAS"]

scaler_knn_ = StandardScaler()
X_knn_ = scaler_knn_.fit_transform(X_knn_)

X_train_knn_, X_test_knn_, y_train_knn_, y_test_knn_ = train_test_split(X_knn_, y_knn_, test_size=0.2, random_state=0)

In [37]:
knn_reg_ = KNeighborsRegressor(n_neighbors=3)
knn_reg_.fit(X_train_knn_, y_train_knn_)

KNeighborsRegressor(n_neighbors=3)

In [38]:
os.environ['OMP_NUM_THREADS'] = '1'
y_pred_knn_ = knn_reg_.predict(X_test_knn_)

mae_knn_ = metrics.mean_absolute_error(y_test_knn_, y_pred_knn_)
mse_knn_ = metrics.mean_squared_error(y_test_knn_, y_pred_knn_)
rmse_knn_ = np.sqrt(metrics.mean_squared_error(y_test_knn_, y_pred_knn_))
r_2_knn_ = r2_score(y_test_knn_, y_pred_knn_)

print('Mean Absolute Error:', mae_knn_)
print('Mean Square Error:', mse_knn_)
print('Root Mean Square Error:', rmse_knn_)
print('R^2:', r_2_knn_)

Mean Absolute Error: 0.47969735764598104
Mean Square Error: 0.5967231530970133
Root Mean Square Error: 0.7724785777592886
R^2: 0.9720379111134039


In [298]:
param_grid_knn = {
    'n_neighbors' : [2, 3, 5, 10], 
}

halving_grid_search_knn = HalvingGridSearchCV(
    knn_reg_, 
    param_grid_knn, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_knn.fit(X_train_knn_, y_train_knn_)


print(f"Best Hyperparameters: {halving_grid_search_knn.best_params_}")
print(f"Best Score: {halving_grid_search_knn.best_score_}")

Best Hyperparameters: {'n_neighbors': 3}
Best Score: 0.9844458663325296


#### ElasticNet

In [39]:
X_en_ = yield_zones_[feature_names]
y_en_ = yield_zones_["VRYIELDMAS"]

scaler = StandardScaler()
X_en_ = scaler.fit_transform(X_en_)

X_train_en_, X_test_en_, y_train_en_, y_test_en_ = train_test_split(X_en_, y_en_, test_size=0.2, random_state=0)

In [40]:
regr_en_ = ElasticNet(alpha=1, tol=1e-4)
regr_en_.fit(X_train_en_, y_train_en_)

ElasticNet(alpha=1)

In [41]:
y_pred_en_ = regr_en_.predict(X_test_en_)

mae_en_ = metrics.mean_absolute_error(y_test_en_, y_pred_en_)
mse_en_ = metrics.mean_squared_error(y_test_en_, y_pred_en_)
rmse_en_ = np.sqrt(metrics.mean_squared_error(y_test_en_, y_pred_en_))
r_2_en_ = r2_score(y_test_en_, y_pred_en_)

print('Mean Absolute Error:', mae_en_)
print('Mean Square Error:', mse_en_)
print('Root Mean Square Error:', rmse_en_)
print('R^2:', r_2_en_)

Mean Absolute Error: 0.9899727787416436
Mean Square Error: 1.5260194151189839
Root Mean Square Error: 1.2353215836853915
R^2: 0.9284916459051973


In [303]:
param_grid_en = {
    'alpha' : [1, 5, 10], 
    'tol' : [1e-4, 1e-6]
}

halving_grid_search_en = HalvingGridSearchCV(
    regr_en_, 
    param_grid_en, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_en.fit(X_train_en_, y_train_en_)


print(f"Best Hyperparameters: {halving_grid_search_en.best_params_}")
print(f"Best Score: {halving_grid_search_en.best_score_}")

Best Hyperparameters: {'alpha': 1, 'tol': 0.0001}
Best Score: 0.9267583435394722


### Gradient Boosting

#### LightGBM

In [42]:
X_gbm_ = yield_zones_[feature_names]
y_gbm_ = yield_zones_["VRYIELDMAS"]

scaler_gbm_weather = StandardScaler()
X_gbm_ = scaler_gbm_weather.fit_transform(X_gbm_)

X_train_gbm_, X_test_gbm_, y_train_gbm_, y_test_gbm_ = train_test_split(X_gbm_, y_gbm_, test_size=0.2, random_state=0)

In [43]:
model_gbm_ = LGBMRegressor(learning_rate=0.1, max_depth=10, n_estimators=300, num_leaves=31)
model_gbm_.fit(X_train_gbm_, y_train_gbm_)

c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor(max_depth=10, n_estimators=300)

In [44]:
y_pred_gbm_ = model_gbm_.predict(X_test_gbm_)

mae_gbm_ = metrics.mean_absolute_error(y_test_gbm_, y_pred_gbm_)
mse_gbm_ = metrics.mean_squared_error(y_test_gbm_, y_pred_gbm_)
rmse_gbm_ = np.sqrt(metrics.mean_squared_error(y_test_gbm_, y_pred_gbm_))
r_2_gbm_ = r2_score(y_test_gbm_, y_pred_gbm_)

print('Mean Absolute Error:', mae_gbm_)
print('Mean Square Error:', mse_gbm_)
print('Root Mean Square Error:', rmse_gbm_)
print('R^2:', r_2_gbm_)

Mean Absolute Error: 0.37424312808101695
Mean Square Error: 0.3580081627288235
Root Mean Square Error: 0.5983378332755029
R^2: 0.9832239523196064


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [307]:
param_grid_gbm = {
    'num_leaves' : [31, 64], 
    'max_depth' : [6, 8, 10], 
    'learning_rate' : [0.01, 0.1], 
    'n_estimators' : [100, 300], 
}

halving_grid_search_gbm = HalvingGridSearchCV(
    model_gbm_, 
    param_grid_gbm, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_gbm.fit(X_train_gbm_, y_train_gbm_)


print(f"Best Hyperparameters: {halving_grid_search_gbm.best_params_}")
print(f"Best Score: {halving_grid_search_gbm.best_score_}")

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 300, 'num_leaves': 31}
Best Score: 0.9888673033157847


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [99]:
filename = './models/light_gbm_weather.sav'
gbm_scaler = "./models/scaler_light_gbm_weather.pkl"
joblib.dump(model_gbm_, filename)
joblib.dump(scaler_gbm_weather, gbm_scaler)

['./models/scaler_light_gbm_weather.pkl']

#### XGB

In [45]:
X_xgb_ = yield_zones_[feature_names]
y_xgb_ = yield_zones_["VRYIELDMAS"]

scaler_xgb_ = StandardScaler()
X_xgb_ = scaler_xgb_.fit_transform(X_xgb_)

X_train_xgb_, X_test_xgb_, y_train_xgb_, y_test_xgb_ = train_test_split(X_xgb_, y_xgb_, test_size=0.2, random_state=0)

In [46]:
regressor_ = xgb.XGBRegressor(learning_rate = 0.01,
                           n_estimators  = 500,
                           max_depth     = 8,
                           eval_metric   = 'rmse')

regressor_.fit(X_train_xgb_, y_train_xgb_)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [47]:
y_pred_xgb_ = regressor_.predict(X_test_xgb_)

mae_xgb_ = metrics.mean_absolute_error(y_test_xgb_, y_pred_xgb_)
mse_xgb_ = metrics.mean_squared_error(y_test_xgb_, y_pred_xgb_)
rmse_xgb_ = np.sqrt(metrics.mean_squared_error(y_test_xgb_, y_pred_xgb_))
r_2_xgb_ = r2_score(y_test_xgb_, y_pred_xgb_)

print('Mean Absolute Error:', mae_xgb_)
print('Mean Square Error:', mse_xgb_)
print('Root Mean Square Error:', rmse_xgb_)
print('R^2:', r_2_xgb_)

Mean Absolute Error: 0.4165235140750866
Mean Square Error: 0.394310031533307
Root Mean Square Error: 0.6279411051470568
R^2: 0.9815228685305959


In [311]:
param_grid_xgb = {
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.001],
    'n_estimators' : [300, 500, 700], 
    'eval_metric' : ['rmse']
    }

grid_search_xgb = HalvingGridSearchCV(
    regressor_, 
    param_grid_xgb, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)

grid_search_xgb.fit(X_train_xgb_, y_train_xgb_)

print("Best set of hyperparameters: ", grid_search_xgb.best_params_)
print("Best score: ", grid_search_xgb.best_score_)

Best set of hyperparameters:  {'eval_metric': 'rmse', 'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 500}
Best score:  0.9879907354639235


## Fields with Weather without year

In [18]:
feature_names_4 = ["area", "Organic_M", "Ca", "Mg", "Mn", "B",
                 "Cu", "Mo", "Fe", "Zn", "S", "P", "K", "Na", "pH", "C.E.C",
                 "Crop_22", "Crop_34", "Crop_41", "Crop_45", "Crop_174", "Crop_22",

    "mean_tempmax_month_5", "mean_tempmax_month_6", "mean_tempmax_month_7",
    "mean_tempmax_month_8", "mean_tempmax_month_9", "mean_tempmax_month_10",
    "std_tempmax_month_5", "std_tempmax_month_6", "std_tempmax_month_7",
    "std_tempmax_month_8", "std_tempmax_month_9", "std_tempmax_month_10",

    "mean_tempmin_month_5", "mean_tempmin_month_6", "mean_tempmin_month_7",
    "mean_tempmin_month_8", "mean_tempmin_month_9", "mean_tempmin_month_10",
    "std_tempmin_month_5", "std_tempmin_month_6", "std_tempmin_month_7",
    "std_tempmin_month_8", "std_tempmin_month_9", "std_tempmin_month_10",

    "mean_temp_month_5", "mean_temp_month_6", "mean_temp_month_7",
    "mean_temp_month_8", "mean_temp_month_9", "mean_temp_month_10",
    "std_temp_month_5", "std_temp_month_6", "std_temp_month_7",
    "std_temp_month_8", "std_temp_month_9", "std_temp_month_10",
    
    "mean_humidity_month_5", "mean_humidity_month_6", "mean_humidity_month_7",
    "mean_humidity_month_8", "mean_humidity_month_9", "mean_humidity_month_10",
    "std_humidity_month_5", "std_humidity_month_6", "std_humidity_month_7",
    "std_humidity_month_8", "std_humidity_month_9", "std_humidity_month_10",
    
    "mean_precip_month_5", "mean_precip_month_6", "mean_precip_month_7",
    "mean_precip_month_8", "mean_precip_month_9", "mean_precip_month_10",
    "std_precip_month_5", "std_precip_month_6", "std_precip_month_7",
    "std_precip_month_8", "std_precip_month_9", "std_precip_month_10",
    
    "mean_cloudcover_month_5", "mean_cloudcover_month_6", "mean_cloudcover_month_7",
    "mean_cloudcover_month_8", "mean_cloudcover_month_9", "mean_cloudcover_month_10",
    "std_cloudcover_month_5", "std_cloudcover_month_6", "std_cloudcover_month_7",
    "std_cloudcover_month_8", "std_cloudcover_month_9", "std_cloudcover_month_10",
    ]

In [19]:
len(feature_names_4)

94

### Regressions

#### MLR

In [20]:
X_mlr_4 = yield_zones_[feature_names_4]
y_mlr_4 = yield_zones_["VRYIELDMAS"]

scaler_mlr_4 = StandardScaler()
X_mlr_4 = scaler_mlr_4.fit_transform(X_mlr_4)

X_train_mlr_4, X_test_mlr_4, y_train_mlr_4, y_test_mlr_4 = train_test_split(X_mlr_4, y_mlr_4, test_size=0.2, random_state=0)

In [22]:
reg_model_mlr_4 = LinearRegression(copy_X=True, fit_intercept=True, n_jobs=15, positive=True)
reg_model_mlr_4 = LinearRegression().fit(X_train_mlr_4, y_train_mlr_4)
y_pred_mlr_4 = reg_model_mlr_4.predict(X_test_mlr_4)  
# print("Prediction for test set: {}".format(y_pred_mlr_4))

In [23]:
mae_mlr_4 = metrics.mean_absolute_error(y_test_mlr_4, y_pred_mlr_4)
mse_mlr_4 = metrics.mean_squared_error(y_test_mlr_4, y_pred_mlr_4)
rmse_mlr_4 = np.sqrt(metrics.mean_squared_error(y_test_mlr_4, y_pred_mlr_4))
r_2_mlr_4 = r2_score(y_test_mlr_4, y_pred_mlr_4)

print('Mean Absolute Error:', mae_mlr_4)
print('Mean Square Error:', mse_mlr_4)
print('Root Mean Square Error:', rmse_mlr_4)
print('R^2:', r_2_mlr_4)

Mean Absolute Error: 0.448473656036082
Mean Square Error: 0.3940060597058502
Root Mean Square Error: 0.6276990199975225
R^2: 0.9815371124680303


#### Lasso

In [24]:
X_lasso_4 = yield_zones_[feature_names_4]
y_lasso_4 = yield_zones_["VRYIELDMAS"]

scaler_lasso_4 = StandardScaler()
X_lasso_4 = scaler_lasso_4.fit_transform(X_lasso_4)

X_train_lasso_4, X_test_lasso_4, y_train_lasso_4, y_test_lasso_4 = train_test_split(X_lasso_4, y_lasso_4, test_size=0.2, random_state=0)

In [25]:
lasso_model_4 = Lasso(alpha=0.1)
lasso_model_4.fit(X_train_lasso_4, y_train_lasso_4)

Lasso(alpha=0.1)

In [26]:
y_pred_lasso_4 = lasso_model_4.predict(X_test_lasso_4)

mae_lasso_4 = metrics.mean_absolute_error(y_test_lasso_4, y_pred_lasso_4)
mse_lasso_4 = metrics.mean_squared_error(y_test_lasso_4, y_pred_lasso_4)
rmse_lasso_4 = np.sqrt(metrics.mean_squared_error(y_test_lasso_4, y_pred_lasso_4))
r_2_lasso_4 = r2_score(y_test_lasso_4, y_pred_lasso_4)

print('Mean Absolute Error:', mae_lasso_4)
print('Mean Square Error:', mse_lasso_4)
print('Root Mean Square Error:', rmse_lasso_4)
print('R^2:', r_2_lasso_4)

Mean Absolute Error: 0.44101768044465733
Mean Square Error: 0.41572792584027557
Root Mean Square Error: 0.6447696688277726
R^2: 0.9805192388553153


#### Ridge

In [27]:
X_ridge_4 = yield_zones_[feature_names_4]
y_ridge_4 = yield_zones_["VRYIELDMAS"]

scaler_ridge_4 = StandardScaler()
X_ridge_4 = scaler_ridge_4.fit_transform(X_ridge_4)

X_train_ridge_4, X_test_ridge_4, y_train_ridge_4, y_test_ridge_4 = train_test_split(X_ridge_4, y_ridge_4, test_size=0.2, random_state=0)

In [28]:
ridge_model_4 = Ridge(alpha=5)
ridge_model_4.fit(X_train_ridge_4, y_train_ridge_4)

Ridge(alpha=5)

In [29]:
y_pred_ridge_4 = ridge_model_4.predict(X_test_ridge_4)

mae_ridge_4 = metrics.mean_absolute_error(y_test_ridge_4, y_pred_ridge_4)
mse_ridge_4 = metrics.mean_squared_error(y_test_ridge_4, y_pred_ridge_4)
rmse_ridge_4 = np.sqrt(metrics.mean_squared_error(y_test_ridge_4, y_pred_ridge_4))
r_2_ridge_4 = r2_score(y_test_ridge_4, y_pred_ridge_4)

print('Mean Absolute Error:', mae_ridge_4)
print('Mean Square Error:', mse_ridge_4)
print('Root Mean Square Error:', rmse_ridge_4)
print('R^2:', r_2_ridge_4)

Mean Absolute Error: 0.43278035400736514
Mean Square Error: 0.38355451229586845
Root Mean Square Error: 0.6193177797349826
R^2: 0.9820268657081445


In [30]:
filename = './models/ridge_weather_.sav'
ridge_scaler = "./models/scaler_ridge_weather_.pkl"
joblib.dump(ridge_model_4, filename)
joblib.dump(scaler_ridge_4, ridge_scaler)

['./models/scaler_ridge_weather_.pkl']

#### Polynomial

In [31]:
X_pol_4 = yield_zones_[feature_names_4]
y_pol_4 = yield_zones_["VRYIELDMAS"]

scaler_pol_4 = StandardScaler()
X_pol_4 = scaler_pol_4.fit_transform(X_pol_4)

X_train_pol_4, X_test_pol_4, y_train_pol_4, y_test_pol_4 = train_test_split(X_pol_4, y_pol_4, test_size=0.2, random_state=0)

In [32]:
lin_pol_4 = LinearRegression()
lin_pol_4.fit(X_train_pol_4, y_train_pol_4)

poly_4 = PolynomialFeatures(degree=3)
X_poly_4 = poly_4.fit_transform(X_train_pol_4)
X_poly_test_4 = poly_4.fit_transform(X_test_pol_4)

poly_4.fit(X_poly_4, y_train_pol_4)
lin2_pol_4 = LinearRegression()
lin2_pol_4.fit(X_poly_4, y_train_pol_4)

LinearRegression()

In [33]:
y_pred_pol_4 = lin2_pol_4.predict(X_poly_test_4)

mae_pol_4 = metrics.mean_absolute_error(y_test_pol_4, y_pred_pol_4)
mse_pol_4 = metrics.mean_squared_error(y_test_pol_4, y_pred_pol_4)
rmse_pol_4 = np.sqrt(metrics.mean_squared_error(y_test_pol_4, y_pred_pol_4))
r_2_pol_4 = r2_score(y_test_pol_4, y_pred_pol_4)

print('Mean Absolute Error:', mae_pol_4)
print('Mean Square Error:', mse_pol_4)
print('Root Mean Square Error:', rmse_pol_4)
print('R^2:', r_2_pol_4)

Mean Absolute Error: 0.8456356203714096
Mean Square Error: 1.1216585344440222
Root Mean Square Error: 1.0590838184223297
R^2: 0.9474397541343033


In [34]:
mae_ = []
mse_ = []
rmse_ = []
r2_ = []

degrees = [2, 3, 4]

for degree in degrees:
    pipeline = Pipeline([('poly_features', PolynomialFeatures(degree=degree)), ('model', LinearRegression())])
    pipeline.fit(X_train_pol_4, y_train_pol_4)

    y_pred_test_ = pipeline.predict(X_test_pol_4)

    mae_.append(metrics.mean_absolute_error(y_test_pol_4, y_pred_pol_4))
    mse_.append(metrics.mean_squared_error(y_test_pol_4, y_pred_pol_4))
    rmse_.append(np.sqrt(metrics.mean_squared_error(y_test_pol_4, y_pred_pol_4)))
    r2_.append(r2_score(y_test_pol_4, y_pred_test_))


In [35]:
print(degrees,'\n')
print("MAE: ", mae_,'\n')
print("MSE: ", mse_, "\n")
print("RMSE: ", rmse_, "\n")
print("R2: ", r2_, "\n")

[2, 3, 4] 

MAE:  [0.8456356203714096, 0.8456356203714096, 0.8456356203714096] 

MSE:  [1.1216585344440222, 1.1216585344440222, 1.1216585344440222] 

RMSE:  [1.0590838184223297, 1.0590838184223297, 1.0590838184223297] 

R2:  [-22.672459329284933, 0.9474397541343033, 0.9641475354895537] 



#### DT

In [36]:
X_dt_4 = yield_zones_[feature_names_4]
y_dt_4 = yield_zones_["VRYIELDMAS"]

scaler_dt_4 = StandardScaler()
X_dt_4 = scaler_dt_4.fit_transform(X_dt_4)

X_train_dt_4, X_test_dt_4, y_train_dt_4, y_test_dt_4 = train_test_split(X_dt_4, y_dt_4, test_size=0.2, random_state=0)

In [37]:
regressor_dt_4 = DecisionTreeRegressor(max_depth=6, max_leaf_nodes=64, random_state=42)
regressor_dt_4.fit(X_train_dt_4, y_train_dt_4)

DecisionTreeRegressor(max_depth=6, max_leaf_nodes=64, random_state=42)

In [38]:
y_pred_dt_4 = regressor_dt_4.predict(X_test_dt_4)

mae_dt_4 = metrics.mean_absolute_error(y_test_dt_4, y_pred_dt_4)
mse_dt_4 = metrics.mean_squared_error(y_test_dt_4, y_pred_dt_4)
rmse_dt_4 = np.sqrt(metrics.mean_squared_error(y_test_dt_4, y_pred_dt_4))
r_2_dt_4 = r2_score(y_test_dt_4, y_pred_dt_4)

print('Mean Absolute Error:', mae_dt_4)
print('Mean Square Error:', mse_dt_4)
print('Root Mean Square Error:', rmse_dt_4)
print('R^2:', r_2_dt_4)

Mean Absolute Error: 0.5180141395110153
Mean Square Error: 0.5519344704305325
Root Mean Square Error: 0.7429229236135687
R^2: 0.9741366819074209


#### RF

In [39]:
X_rf_4 = yield_zones_[feature_names_4]
y_rf_4 = yield_zones_["VRYIELDMAS"]

scaler_rf_4 = StandardScaler()
X_rf_4 = scaler_rf_4.fit_transform(X_rf_4)

X_train_rf_4, X_test_rf_4, y_train_rf_4, y_test_rf_4 = train_test_split(X_rf_4, y_rf_4, test_size=0.2, random_state=0)

In [40]:
regr_rf_4 = RandomForestRegressor(max_depth=10, max_leaf_nodes=32, n_estimators=100)
regr_rf_4.fit(X_train_rf_4, y_train_rf_4)

RandomForestRegressor(max_depth=10, max_leaf_nodes=32)

In [41]:
y_pred_rf_4 = regr_rf_4.predict(X_test_rf_4)

mae_rf_4 = metrics.mean_absolute_error(y_test_rf_4, y_pred_rf_4)
mse_rf_4 = metrics.mean_squared_error(y_test_rf_4, y_pred_rf_4)
rmse_rf_4 = np.sqrt(metrics.mean_squared_error(y_test_rf_4, y_pred_rf_4))
r_2_rf_4 = r2_score(y_test_rf_4, y_pred_rf_4)

print('Mean Absolute Error:', mae_rf_4)
print('Mean Square Error:', mse_rf_4)
print('Root Mean Square Error:', rmse_rf_4)
print('R^2:', r_2_rf_4)

Mean Absolute Error: 0.4336585059862454
Mean Square Error: 0.4787808849205745
Root Mean Square Error: 0.6919399431457722
R^2: 0.9775646150281424


#### KNN

In [42]:
X_knn_4 = yield_zones_[feature_names_4]
y_knn_4 = yield_zones_["VRYIELDMAS"]

scaler_knn_4 = StandardScaler()
X_knn_4 = scaler_knn_4.fit_transform(X_knn_4)

X_train_knn_4, X_test_knn_4, y_train_knn_4, y_test_knn_4 = train_test_split(X_knn_4, y_knn_4, test_size=0.2, random_state=0)

In [43]:
knn_reg_4 = KNeighborsRegressor(n_neighbors=3)
knn_reg_4.fit(X_train_knn_4, y_train_knn_4)

KNeighborsRegressor(n_neighbors=3)

In [44]:
os.environ['OMP_NUM_THREADS'] = '1'
y_pred_knn_4 = knn_reg_4.predict(X_test_knn_4)

mae_knn_4 = metrics.mean_absolute_error(y_test_knn_4, y_pred_knn_4)
mse_knn_4 = metrics.mean_squared_error(y_test_knn_4, y_pred_knn_4)
rmse_knn_4 = np.sqrt(metrics.mean_squared_error(y_test_knn_4, y_pred_knn_4))
r_2_knn_4 = r2_score(y_test_knn_4, y_pred_knn_4)

print('Mean Absolute Error:', mae_knn_4)
print('Mean Square Error:', mse_knn_4)
print('Root Mean Square Error:', rmse_knn_4)
print('R^2:', r_2_knn_4)

Mean Absolute Error: 0.47969735764598104
Mean Square Error: 0.5967231530970133
Root Mean Square Error: 0.7724785777592886
R^2: 0.9720379111134039


#### Elastic Net

In [45]:
X_en_4 = yield_zones_[feature_names_4]
y_en_4 = yield_zones_["VRYIELDMAS"]

scaler_en_4 = StandardScaler()
X_en_4 = scaler_en_4.fit_transform(X_en_4)

X_train_en_4, X_test_en_4, y_train_en_4, y_test_en_4 = train_test_split(X_en_4, y_en_4, test_size=0.2, random_state=0)

In [46]:
regr_en_4 = ElasticNet(alpha=1, tol=1e-4)
regr_en_4.fit(X_train_en_4, y_train_en_4)

ElasticNet(alpha=1)

In [48]:
y_pred_en_4 = regr_en_4.predict(X_test_en_4)

mae_en_4 = metrics.mean_absolute_error(y_test_en_4, y_pred_en_4)
mse_en_4 = metrics.mean_squared_error(y_test_en_4, y_pred_en_4)
rmse_en_4 = np.sqrt(metrics.mean_squared_error(y_test_en_4, y_pred_en_4))
r_2_en_4 = r2_score(y_test_en_4, y_pred_en_4)

print('Mean Absolute Error:', mae_en_4)
print('Mean Square Error:', mse_en_4)
print('Root Mean Square Error:', rmse_en_4)
print('R^2:', r_2_en_4)

Mean Absolute Error: 0.9899834365040634
Mean Square Error: 1.52603956160233
Root Mean Square Error: 1.2353297380061445
R^2: 0.9284907018530767


### Gradient Boosting

#### LightGBM

In [49]:
X_gbm_4 = yield_zones_[feature_names_4]
y_gbm_4 = yield_zones_["VRYIELDMAS"]

scaler_gbm_weather_4 = StandardScaler()
X_gbm_4 = scaler_gbm_weather_4.fit_transform(X_gbm_4)

X_train_gbm_4, X_test_gbm_4, y_train_gbm_4, y_test_gbm_4 = train_test_split(X_gbm_4, y_gbm_4, test_size=0.2, random_state=0)

In [50]:
model_gbm_4 = LGBMRegressor(learning_rate=0.1, max_depth=10, n_estimators=300, num_leaves=31)
model_gbm_4.fit(X_train_gbm_4, y_train_gbm_4)

c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor(max_depth=10, n_estimators=300)

In [51]:
y_pred_gbm_4 = model_gbm_4.predict(X_test_gbm_4)

mae_gbm_4 = metrics.mean_absolute_error(y_test_gbm_4, y_pred_gbm_4)
mse_gbm_4 = metrics.mean_squared_error(y_test_gbm_4, y_pred_gbm_4)
rmse_gbm_4 = np.sqrt(metrics.mean_squared_error(y_test_gbm_4, y_pred_gbm_4))
r_2_gbm_4 = r2_score(y_test_gbm_4, y_pred_gbm_4)

print('Mean Absolute Error:', mae_gbm_4)
print('Mean Square Error:', mse_gbm_4)
print('Root Mean Square Error:', rmse_gbm_4)
print('R^2:', r_2_gbm_4)

Mean Absolute Error: 0.39761403882275853
Mean Square Error: 0.38007525555570465
Root Mean Square Error: 0.6165024375910485
R^2: 0.9821899015912384


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [56]:
# filename = './models/light_gbm_weather_.sav'
# gbm_scaler = "./models/scaler_light_gbm_weather_.pkl"
# joblib.dump(model_gbm_4, filename)
# joblib.dump(scaler_gbm_weather_4, gbm_scaler)

#### XGBoost

In [53]:
X_xgb_4 = yield_zones_[feature_names_4]
y_xgb_4 = yield_zones_["VRYIELDMAS"]

scaler_xgb_4 = StandardScaler()
X_xgb_4 = scaler_xgb_4.fit_transform(X_xgb_4)

X_train_xgb_4, X_test_xgb_4, y_train_xgb_4, y_test_xgb_4 = train_test_split(X_xgb_4, y_xgb_4, test_size=0.2, random_state=0)

In [54]:
regressor_4 = xgb.XGBRegressor(learning_rate = 0.01,
                           n_estimators  = 500,
                           max_depth     = 8,
                           eval_metric   = 'rmse')

regressor_4.fit(X_train_xgb_4, y_train_xgb_4)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [55]:
y_pred_xgb_4 = regressor_4.predict(X_test_xgb_4)

mae_xgb_4 = metrics.mean_absolute_error(y_test_xgb_4, y_pred_xgb_4)
mse_xgb_4 = metrics.mean_squared_error(y_test_xgb_4, y_pred_xgb_4)
rmse_xgb_4 = np.sqrt(metrics.mean_squared_error(y_test_xgb_4, y_pred_xgb_4))
r_2_xgb_4 = r2_score(y_test_xgb_4, y_pred_xgb_4)

print('Mean Absolute Error:', mae_xgb_4)
print('Mean Square Error:', mse_xgb_4)
print('Root Mean Square Error:', rmse_xgb_4)
print('R^2:', r_2_xgb_4)

Mean Absolute Error: 0.404840805592627
Mean Square Error: 0.36976398192844295
Root Mean Square Error: 0.6080822164217952
R^2: 0.9826730816860664


In [57]:
filename = './models/xgb_weather_.sav'
xgb_scaler = "./models/scaler_xgb_weather_.pkl"
joblib.dump(regressor_4, filename)
joblib.dump(scaler_xgb_4, xgb_scaler)

['./models/scaler_xgb_weather_.pkl']

## Fields without weather

In [48]:
all_yields_df_60_ = pd.read_csv("./data_1/all_yields_60.csv")
all_yields_df_62_ = pd.read_csv("./data_1/all_yields.csv")

In [49]:
yield_df_ = pd.concat([all_yields_df_60_, all_yields_df_62_], axis=0)

In [50]:
yield_df_

,VRYIELDMAS,SECTIONID,Moisture,Elevation,area,Organic M.,Ca,Mg,Mn,B,...,year,Crop_33,Crop_34,Crop_41,Crop_45,Crop_174,xcoord,ycoord,Organic M,Crop_22
0,1.384943,1716,13.07,296.027082,4.70,4.7,4001.0,158.0,193.0,3.31,...,2017,True,False,False,False,False,NaN,NaN,NaN,NaN
1,1.492232,1716,13.13,296.047082,4.70,4.7,4001.0,158.0,193.0,3.31,...,2017,True,False,False,False,False,NaN,NaN,NaN,NaN
2,1.655677,1716,13.14,296.041082,4.70,4.7,4001.0,158.0,193.0,3.31,...,2017,True,False,False,False,False,NaN,NaN,NaN,NaN
3,1.825828,1716,13.15,296.050082,4.70,4.7,4001.0,158.0,193.0,3.31,...,2017,True,False,False,False,False,NaN,NaN,NaN,NaN
4,1.825828,1716,13.17,296.044082,4.70,4.7,4001.0,158.0,193.0,3.31,...,2017,True,False,False,False,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185127,0.281828,2373,12.00,289.456031,4.71,NaN,4779.0,168.0,111.0,2.42,...,2024,NaN,False,False,False,True,25.838831,48.828348,6.0,False
185128,0.153946,2373,12.00,289.436031,4.71,NaN,4779.0,168.0,111.0,2.42,...,2024,NaN,False,False,False,True,25.838806,48.828337,6.0,False
185129,0.135194,2373,12.00,289.436031,4.71,NaN,4779.0,168.0,111.0,2.42,...,2024,NaN,False,False,False,True,25.838781,48.828327,6.0,False
185130,0.113808,2373,12.00,289.463031,4.71,NaN,4779.0,168.0,111.0,2.42,...,2024,NaN,False,False,False,True,25.838757,48.828315,6.0,False


In [51]:
yield_df_['Organic_M'] = yield_df_['Organic M.'].combine_first(yield_df_['Organic M'])

In [52]:
yield_zones_1 = yield_df_.groupby(['area', 'year']).mean().reset_index()

In [53]:
yield_zones_1_ = yield_zones_1.drop(columns=["SECTIONID", 'Organic M.', 'Organic M', 
                                         'xcoord', 'ycoord'])

In [54]:
yield_zones_1_['Crop_22'] = yield_zones_1_['Crop_22'].fillna(False).astype(bool)

In [145]:
# yield_zones_1_

In [55]:
yield_zones_1_['Crop_22'] = yield_zones_1_['Crop_22'].fillna(False).astype(bool)
yield_zones_1_['Crop_33'] = yield_zones_1_["Crop_33"].fillna(False).astype(bool)
yield_zones_1_['Crop_34'] = yield_zones_1_['Crop_34'].astype(bool)
yield_zones_1_['Crop_41'] = yield_zones_1_['Crop_41'].astype(bool)
yield_zones_1_['Crop_45'] = yield_zones_1_['Crop_45'].astype(bool)
yield_zones_1_['Crop_174'] = yield_zones_1_['Crop_174'].astype(bool)

In [56]:
yield_zones_1_.isna().sum()

area          0
year          0
VRYIELDMAS    0
Moisture      0
Elevation     0
Ca            0
Mg            0
Mn            0
B             0
Cu            0
Mo            0
Fe            0
Zn            0
S             0
P             0
K             0
Na            0
pH            0
C.E.C         0
Crop_33       0
Crop_34       0
Crop_41       0
Crop_45       0
Crop_174      0
Crop_22       0
Organic_M     0
dtype: int64

In [13]:
yield_zones_1_.columns

Index(['area', 'year', 'VRYIELDMAS', 'Moisture', 'Elevation', 'Ca', 'Mg', 'Mn',
       'B', 'Cu', 'Mo', 'Fe', 'Zn', 'S', 'P', 'K', 'Na', 'pH', 'C.E.C',
       'Crop_33', 'Crop_34', 'Crop_41', 'Crop_45', 'Crop_174', 'Crop_22',
       'Organic_M'],
      dtype='object')

In [57]:
yield_zones_1_.to_csv("./data_1/yield_zones_withot_weather.csv")

In [58]:
features = ['Crop_33', 'Crop_34', 'Crop_41', 'Crop_45', 'Crop_174', 'Crop_22',
          'Elevation', 'area', 'Organic_M', 'Ca', 'Mg', 'Mn',
          'B', 'Cu', 'Mo', 'Fe', 'Zn', 'S', 'P', 'K', 'Na', 'pH', 'C.E.C']

### Regressions

#### MLR

In [62]:
X_mlr_2 = yield_zones_1_[features]
y_mlr_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_mlr_2 = scaler.fit_transform(X_mlr_2)

X_train_mlr_2, X_test_mlr_2, y_train_mlr_2, y_test_mlr_2 = train_test_split(X_mlr_2, y_mlr_2, test_size=0.2, random_state=0)

In [60]:
reg_model_mlr_2 = LinearRegression(copy_X=True, fit_intercept=True, n_jobs=15, positive=True)
reg_model_mlr_2 = LinearRegression().fit(X_train_mlr_2, y_train_mlr_2)
print('Intercept: ', reg_model_mlr_2.intercept_)
y_pred_mlr_2 = reg_model_mlr_2.predict(X_test_mlr_2)  
print("Prediction for test set: {}".format(y_pred_mlr_2))

Intercept:  5.348100435190118
Prediction for test set: [ 1.09293342  3.29902697  1.65854234 12.71944948 12.36033876 12.41768095
  1.94611595  0.53377888 12.78691382 12.98643778 13.34203234  3.33749131
  2.90380866  1.06423352  2.64769676  2.56594733  1.27565171 11.94218555
  3.25547913 10.28219408  3.30217752  1.97603844  3.26309387  0.99406452
  3.63873442  1.24164728 11.15851728  2.64206944 12.24837797  2.09671649
  2.84013406  3.05692679  3.63770456  2.37620573 12.46917833  2.67991352
  1.28101469  1.49114695  2.87955622  2.97884569 12.86355352  2.85247632
  3.91159088  3.32514405  3.18497946  3.26367341]


In [61]:
mae_mlr_2 = metrics.mean_absolute_error(y_test_mlr_2, y_pred_mlr_2)
mse_mlr_2 = metrics.mean_squared_error(y_test_mlr_2, y_pred_mlr_2)
rmse_mlr_2 = np.sqrt(metrics.mean_squared_error(y_test_mlr_2, y_pred_mlr_2))
r_2_mlr_2 = r2_score(y_test_mlr_2, y_pred_mlr_2)

print('Mean Absolute Error:', mae_mlr_2)
print('Mean Square Error:', mse_mlr_2)
print('Root Mean Square Error:', rmse_mlr_2)
print('R^2:', r_2_mlr_2)

Mean Absolute Error: 0.8677376531892453
Mean Square Error: 1.6595808061417299
Root Mean Square Error: 1.2882471836343092
R^2: 0.922233039266235


In [ ]:
param_grid = {
    'copy_X': [True,False], 
    'fit_intercept': [True,False], 
    'n_jobs': [1, 5, 10, 15, None], 
    'positive': [True,False]
    }

halving_grid_search_mlr = HalvingGridSearchCV(
    reg_model_mlr_2, 
    param_grid, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
    )
halving_grid_search_mlr.fit(X_train_mlr_2, y_train_mlr_2)

print(f"Best Hyperparameters: {halving_grid_search_mlr.best_params_}")
print(f"Best Score: {halving_grid_search_mlr.best_score_}")

Best Hyperparameters: {'copy_X': True, 'fit_intercept': True, 'n_jobs': 15, 'positive': True}
Best Score: 0.9137123689371404

#### Lasso

In [29]:
X_lasso_2 = yield_zones_1_[features]
y_lasso_2 = yield_zones_1_["VRYIELDMAS"]

scaler_lasso = StandardScaler()
X_lasso_2 = scaler_lasso.fit_transform(X_lasso_2)

X_train_lasso_2, X_test_lasso_2, y_train_lasso_2, y_test_lasso_2 = train_test_split(X_lasso_2, y_lasso_2, test_size=0.2, random_state=0)

In [30]:
lasso_model_2 = Lasso(alpha=0.1)
lasso_model_2.fit(X_train_lasso_2, y_train_lasso_2)

Lasso(alpha=0.1)

In [31]:
y_pred_lasso_2 = lasso_model_2.predict(X_test_lasso_2)

mae_lasso_2 = metrics.mean_absolute_error(y_test_lasso_2, y_pred_lasso_2)
mse_lasso_2 = metrics.mean_squared_error(y_test_lasso_2, y_pred_lasso_2)
rmse_lasso_2 = np.sqrt(metrics.mean_squared_error(y_test_lasso_2, y_pred_lasso_2))
r_2_lasso_2 = r2_score(y_test_lasso_2, y_pred_lasso_2)

print('Mean Absolute Error:', mae_lasso_2)
print('Mean Square Error:', mse_lasso_2)
print('Root Mean Square Error:', rmse_lasso_2)
print('R^2:', r_2_lasso_2)

Mean Absolute Error: 0.8208157212454141
Mean Square Error: 1.4570854772829664
Root Mean Square Error: 1.2070979567884979
R^2: 0.9317218488679445


In [32]:
param_grid_lasso = {
    'alpha' : [0.1, 1, 5, 10, 20, 50, 100]
    }

grid_search_lasso = HalvingGridSearchCV(
    lasso_model_2, 
    param_grid_lasso, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
    )
grid_search_lasso.fit(X_train_lasso_2, y_train_lasso_2)

print(f"Best Hyperparameters: {grid_search_lasso.best_params_}")
print(f"Best Score: {grid_search_lasso.best_score_}")

Best Hyperparameters: {'alpha': 0.1}
Best Score: 0.9384822273735844


In [33]:
filename = './models/lasso.sav'
gbm_scaler = "./models/scaler_lasso.pkl"
joblib.dump(lasso_model_2, filename)
joblib.dump(scaler_lasso, gbm_scaler)

['./models/scaler_lasso.pkl']

#### Ridge

In [111]:
X_ridge_2 = yield_zones_1_[features]
y_ridge_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_ridge_2 = scaler.fit_transform(X_ridge_2)

X_train_ridge_2, X_test_ridge_2, y_train_ridge_2, y_test_ridge_2 = train_test_split(X_ridge_2, y_ridge_2, test_size=0.2, random_state=0)

In [112]:
ridge_model_2 = Ridge(alpha=1)
ridge_model_2.fit(X_train_ridge_2, y_train_ridge_2)

Ridge(alpha=1)

In [113]:
y_pred_ridge_2 = ridge_model_2.predict(X_test_ridge_2)
mae_ridge_2 = metrics.mean_absolute_error(y_test_ridge_2, y_pred_ridge_2)
mse_ridge_2 = metrics.mean_squared_error(y_test_ridge_2, y_pred_ridge_2)
rmse_ridge_2 = np.sqrt(metrics.mean_squared_error(y_test_ridge_2, y_pred_ridge_2))
r_2_ridge_2 = r2_score(y_test_ridge_2, y_pred_ridge_2)

print('Mean Absolute Error:', mae_ridge_2)
print('Mean Square Error:', mse_ridge_2)
print('Root Mean Square Error:', rmse_ridge_2)
print('R^2:', r_2_ridge_2)

Mean Absolute Error: 0.8658682057907139
Mean Square Error: 1.5508440237333143
Root Mean Square Error: 1.2453288817550625
R^2: 0.9273283796416943


In [177]:
param_grid_ridge = {
    'alpha' : [0.0001, 0.001, 0.01, 0.1, 1, 5, 10, 20, 50, 100]
    }

grid_search_ridge = HalvingGridSearchCV(
    ridge_model_2, 
    param_grid_ridge, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
    )
grid_search_ridge.fit(X_train_ridge_2, y_train_ridge_2)

print(f"Best Hyperparameters: {grid_search_ridge.best_params_}")
print(f"Best Score: {grid_search_ridge.best_score_}")

Best Hyperparameters: {'alpha': 1}
Best Score: 0.9701102028518228


#### Polynomial

In [63]:
X_pol_2 = yield_zones_1_[features]
y_pol_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_pol_2 = scaler.fit_transform(X_pol_2)

X_train_pol_2, X_test_pol_2, y_train_pol_2, y_test_pol_2 = train_test_split(X_pol_2, y_pol_2, test_size=0.2, random_state=0)

In [64]:
lin_pol_2 = LinearRegression()
lin_pol_2.fit(X_train_pol_2, y_train_pol_2)

poly_2 = PolynomialFeatures(degree=2)
X_poly_2 = poly_2.fit_transform(X_train_pol_2)
X_poly_test_2 = poly_2.fit_transform(X_test_pol_2)

poly_2.fit(X_poly_2, y_train_pol_2)
lin2_pol_2 = LinearRegression()
lin2_pol_2.fit(X_poly_2, y_train_pol_2)

LinearRegression()

In [65]:
y_pred_pol_2 = lin2_pol_2.predict(X_poly_test_2)

mae_pol_2 = metrics.mean_absolute_error(y_test_pol_2, y_pred_pol_2)
mse_pol_2 = metrics.mean_squared_error(y_test_pol_2, y_pred_pol_2)
rmse_pol_2 = np.sqrt(metrics.mean_squared_error(y_test_pol_2, y_pred_pol_2))
r_2_pol_2 = r2_score(y_test_pol_2, y_pred_pol_2)

print('Mean Absolute Error:', mae_pol_2)
print('Mean Square Error:', mse_pol_2)
print('Root Mean Square Error:', rmse_pol_2)
print('R^2:', r_2_pol_2)

Mean Absolute Error: 84.36235230259157
Mean Square Error: 67261.1439521126
Root Mean Square Error: 259.3475350800786
R^2: -3150.816845118022


In [66]:
mae_ = []
mse_ = []
rmse_ = []
r2_ = []

degrees = [2, 3, 4]

for degree in degrees:
    pipeline = Pipeline([('poly_features', PolynomialFeatures(degree=degree)), ('model', LinearRegression())])
    pipeline.fit(X_train_pol_2, y_train_pol_2)

    y_pred_test_ = pipeline.predict(X_test_pol_2)

    mae_.append(metrics.mean_absolute_error(y_test_pol_2, y_pred_pol_2))
    mse_.append(metrics.mean_squared_error(y_test_pol_2, y_pred_pol_2))
    rmse_.append(np.sqrt(metrics.mean_squared_error(y_test_pol_2, y_pred_pol_2)))
    r2_.append(r2_score(y_test_pol_2, y_pred_test_))

print(degrees,'\n')
print("MAE: ", mae_,'\n')
print("MSE: ", mse_, "\n")
print("RMSE: ", rmse_, "\n")
print("R2: ", r2_, "\n")

[2, 3, 4] 

MAE:  [84.36235230259157, 84.36235230259157, 84.36235230259157] 

MSE:  [67261.1439521126, 67261.1439521126, 67261.1439521126] 

RMSE:  [259.3475350800786, 259.3475350800786, 259.3475350800786] 

R2:  [-3150.816845118022, -3290823.7684275974, -339318.4122845205] 



#### Decision Tree

In [120]:
X_dt_2 = yield_zones_1_[features]
y_dt_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_dt_2 = scaler.fit_transform(X_dt_2)

X_train_dt_2, X_test_dt_2, y_train_dt_2, y_test_dt_2 = train_test_split(X_dt_2, y_dt_2, test_size=0.2, random_state=0)

In [121]:
regressor_dt_2 = DecisionTreeRegressor(max_depth=10, max_leaf_nodes=16, random_state=42)
regressor_dt_2.fit(X_train_dt_2, y_train_dt_2)

DecisionTreeRegressor(max_depth=10, max_leaf_nodes=16, random_state=42)

In [122]:
y_pred_dt_2 = regressor_dt_2.predict(X_test_dt_2)

mae_dt_2 = metrics.mean_absolute_error(y_test_dt_2, y_pred_dt_2)
mse_dt_2 = metrics.mean_squared_error(y_test_dt_2, y_pred_dt_2)
rmse_dt_2 = np.sqrt(metrics.mean_squared_error(y_test_dt_2, y_pred_dt_2))
r_2_dt_2 = r2_score(y_test_dt_2, y_pred_dt_2)

print('Mean Absolute Error:', mae_dt_2)
print('Mean Square Error:', mse_dt_2)
print('Root Mean Square Error:', rmse_dt_2)
print('R^2:', r_2_dt_2)

Mean Absolute Error: 1.100707354846934
Mean Square Error: 3.4669319919285915
Root Mean Square Error: 1.8619699224016997
R^2: 0.8375416472128571


In [190]:
param_grid_dt = {
    'max_depth' : [4, 6, 8, 10], 
    'max_leaf_nodes' : [16, 32, 64]
}

halving_grid_search_dt = HalvingGridSearchCV(
    regressor_dt_2, 
    param_grid_dt, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_dt.fit(X_train_dt_2, y_train_dt_2)


print(f"Best Hyperparameters: {halving_grid_search_dt.best_params_}")
print(f"Best Score: {halving_grid_search_dt.best_score_}")

Best Hyperparameters: {'max_depth': 10, 'max_leaf_nodes': 16}
Best Score: 0.9842933663119281


#### Random Forest

In [123]:
X_rf_2 = yield_zones_1_[features]
y_rf_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_rf_2 = scaler.fit_transform(X_rf_2)

X_train_rf_2, X_test_rf_2, y_train_rf_2, y_test_rf_2 = train_test_split(X_rf_2, y_rf_2, test_size=0.2, random_state=0)

In [124]:
regr_rf_2 = RandomForestRegressor(max_depth=10, max_leaf_nodes=32, n_estimators=100)
regr_rf_2.fit(X_train_rf_2, y_train_rf_2)

RandomForestRegressor(max_depth=10, max_leaf_nodes=32)

In [125]:
y_pred_rf_2 = regr_rf_2.predict(X_test_rf_2)

mae_rf_2 = metrics.mean_absolute_error(y_test_rf_2, y_pred_rf_2)
mse_rf_2 = metrics.mean_squared_error(y_test_rf_2, y_pred_rf_2)
rmse_rf_2 = np.sqrt(metrics.mean_squared_error(y_test_rf_2, y_pred_rf_2))
r_2_rf_2 = r2_score(y_test_rf_2, y_pred_rf_2)

print('Mean Absolute Error:', mae_rf_2)
print('Mean Square Error:', mse_rf_2)
print('Root Mean Square Error:', rmse_rf_2)
print('R^2:', r_2_rf_2)

Mean Absolute Error: 1.0076621315203387
Mean Square Error: 2.6984169244509504
Root Mean Square Error: 1.6426858873354182
R^2: 0.8735538021224966


In [202]:
param_grid_rf = {
    'max_depth' : [6, 8, 10], 
    'max_leaf_nodes' : [32, 64], 
    'n_estimators' : [100, 200, 300]
}

halving_grid_search_rf = HalvingGridSearchCV(
    regr_rf_2, 
    param_grid_rf, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_rf.fit(X_train_rf_2, y_train_rf_2)


print(f"Best Hyperparameters: {halving_grid_search_rf.best_params_}")
print(f"Best Score: {halving_grid_search_rf.best_score_}")

Best Hyperparameters: {'max_depth': 10, 'max_leaf_nodes': 32, 'n_estimators': 100}
Best Score: 0.9894903031552046


#### KNN

In [126]:
X_knn_2 = yield_zones_1_[features]
y_knn_2 = yield_zones_1_["VRYIELDMAS"]

scaler_knn_2 = StandardScaler()
X_knn_2 = scaler_knn_2.fit_transform(X_knn_2)

X_train_knn_2, X_test_knn_2, y_train_knn_2, y_test_knn_2 = train_test_split(X_knn_2, y_knn_2, test_size=0.2, random_state=0)

In [127]:
knn_reg_2 = KNeighborsRegressor(n_neighbors=3)
knn_reg_2.fit(X_train_knn_2, y_train_knn_2)

KNeighborsRegressor(n_neighbors=3)

In [128]:
os.environ['OMP_NUM_THREADS'] = '1'
y_pred_knn_2 = knn_reg_2.predict(X_test_knn_2)

mae_knn_2 = metrics.mean_absolute_error(y_test_knn_2, y_pred_knn_2)
mse_knn_2 = metrics.mean_squared_error(y_test_knn_2, y_pred_knn_2)
rmse_knn_2 = np.sqrt(metrics.mean_squared_error(y_test_knn_2, y_pred_knn_2))
r_2_knn_2 = r2_score(y_test_knn_2, y_pred_knn_2)

print('Mean Absolute Error:', mae_knn_2)
print('Mean Square Error:', mse_knn_2)
print('Root Mean Square Error:', rmse_knn_2)
print('R^2:', r_2_knn_2)

Mean Absolute Error: 2.1372607713164813
Mean Square Error: 11.663702808072772
Root Mean Square Error: 3.415216363288389
R^2: 0.45344588529289376


In [208]:
param_grid_knn = {
    'n_neighbors' : [2, 3, 5, 10], 
}

halving_grid_search_knn = HalvingGridSearchCV(
    knn_reg_2, 
    param_grid_knn, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_knn.fit(X_train_knn_2, y_train_knn_2)


print(f"Best Hyperparameters: {halving_grid_search_knn.best_params_}")
print(f"Best Score: {halving_grid_search_knn.best_score_}")

Best Hyperparameters: {'n_neighbors': 3}
Best Score: 0.8316189914600225


#### ElasticNet

In [129]:
X_en_2 = yield_zones_1_[features]
y_en_2 = yield_zones_1_["VRYIELDMAS"]

scaler = StandardScaler()
X_en_2 = scaler.fit_transform(X_en_2)

X_train_en_2, X_test_en_2, y_train_en_2, y_test_en_2 = train_test_split(X_en_2, y_en_2, test_size=0.2, random_state=0)

In [130]:
regr_en_2 = ElasticNet(alpha=1, tol=1e-4)
regr_en_2.fit(X_train_en_2, y_train_en_2)
print(regr_en_2.intercept_)

5.3648251646925385


In [131]:
y_pred_en_2 = regr_en_2.predict(X_test_en_2)

mae_en_2 = metrics.mean_absolute_error(y_test_en_2, y_pred_en_2)
mse_en_2 = metrics.mean_squared_error(y_test_en_2, y_pred_en_2)
rmse_en_2 = np.sqrt(metrics.mean_squared_error(y_test_en_2, y_pred_en_2))
r_2_en_2 = r2_score(y_test_en_2, y_pred_en_2)

print('Mean Absolute Error:', mae_en_2)
print('Mean Square Error:', mse_en_2)
print('Root Mean Square Error:', rmse_en_2)
print('R^2:', r_2_en_2)

Mean Absolute Error: 1.6950838853468055
Mean Square Error: 4.439515357354518
Root Mean Square Error: 2.107015746821679
R^2: 0.7919669743138433


In [212]:
param_grid_en = {
    'alpha' : [1, 5, 10], 
    'tol' : [1e-4, 1e-6]
}

halving_grid_search_en = HalvingGridSearchCV(
    regr_en_2, 
    param_grid_en, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_en.fit(X_train_en_2, y_train_en_2)


print(f"Best Hyperparameters: {halving_grid_search_en.best_params_}")
print(f"Best Score: {halving_grid_search_en.best_score_}")

Best Hyperparameters: {'alpha': 1, 'tol': 0.0001}
Best Score: 0.8165475446769106


### Gradient Boosting

#### LightGBM

In [136]:
X_gbm_2 = yield_zones_1_[features]
y_gbm_2 = yield_zones_1_["VRYIELDMAS"]

scaler_gbm = StandardScaler()
X_gbm_2 = scaler_gbm.fit_transform(X_gbm_2)

X_train_gbm_2, X_test_gbm_2, y_train_gbm_2, y_test_gbm_2 = train_test_split(X_gbm_2, y_gbm_2, test_size=0.2, random_state=0)

In [137]:
model_gbm_2 = LGBMRegressor(learning_rate=0.1, max_depth=6, n_estimators=100, num_leaves=64)
model_gbm_2.fit(X_train_gbm_2, y_train_gbm_2)

c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor(max_depth=6, num_leaves=64)

In [138]:
y_pred_gbm_2 = model_gbm_2.predict(X_test_gbm_2)

mae_gbm_2 = metrics.mean_absolute_error(y_test_gbm_2, y_pred_gbm_2)
mse_gbm_2 = metrics.mean_squared_error(y_test_gbm_2, y_pred_gbm_2)
rmse_gbm_2 = np.sqrt(metrics.mean_squared_error(y_test_gbm_2, y_pred_gbm_2))
r_2_gbm_2 = r2_score(y_test_gbm_2, y_pred_gbm_2)

print('Mean Absolute Error:', mae_gbm_2)
print('Mean Square Error:', mse_gbm_2)
print('Root Mean Square Error:', rmse_gbm_2)
print('R^2:', r_2_gbm_2)

Mean Absolute Error: 0.9376073141059514
Mean Square Error: 2.1641833280400204
Root Mean Square Error: 1.4711163543513546
R^2: 0.8985876678800394


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [230]:
param_grid_gbm = {
    'num_leaves' : [31, 32, 64], 
    'max_depth' : [6, 8, 10], 
    'learning_rate' : [0.1], 
    'n_estimators' : [100, 300], 
}

halving_grid_search_gbm = HalvingGridSearchCV(
    model_gbm_2, 
    param_grid_gbm, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)
halving_grid_search_gbm.fit(X_train_gbm_2, y_train_gbm_2)


print(f"Best Hyperparameters: {halving_grid_search_gbm.best_params_}")
print(f"Best Score: {halving_grid_search_gbm.best_score_}")

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'num_leaves': 64}
Best Score: 0.9853466586978445


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [95]:
filename = './models/light_gbm.sav'
gbm_scaler = "./models/scaler_light_gbm.pkl"
joblib.dump(model_gbm_2, filename)
joblib.dump(scaler_gbm, gbm_scaler)

['./models/scaler_light_gbm.pkl']

#### XGBoost

In [139]:
X_xgb_2 = yield_zones_1_[features]
y_xgb_2 = yield_zones_1_["VRYIELDMAS"]

scaler_xgb_2 = StandardScaler()
X_xgb_2 = scaler_xgb_2.fit_transform(X_xgb_2)

X_train_xgb_2, X_test_xgb_2, y_train_xgb_2, y_test_xgb_2 = train_test_split(X_xgb_2, y_xgb_2, test_size=0.2, random_state=0)

In [140]:
regressor_2 = xgb.XGBRegressor(learning_rate = 0.01,
                           n_estimators  = 500,
                           max_depth     = 10,
                           eval_metric   = 'rmse')

regressor_2.fit(X_train_xgb_2, y_train_xgb_2)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [141]:
y_pred_xgb_2 = regressor_2.predict(X_test_xgb_2)

mae_xgb_2 = metrics.mean_absolute_error(y_test_xgb_2, y_pred_xgb_2)
mse_xgb_2 = metrics.mean_squared_error(y_test_xgb_2, y_pred_xgb_2)
rmse_xgb_2 = np.sqrt(metrics.mean_squared_error(y_test_xgb_2, y_pred_xgb_2))
r_2_xgb_2 = r2_score(y_test_xgb_2, y_pred_xgb_2)

print('Mean Absolute Error:', mae_xgb_2)
print('Mean Square Error:', mse_xgb_2)
print('Root Mean Square Error:', rmse_xgb_2)
print('R^2:', r_2_xgb_2)

Mean Absolute Error: 1.219872236156911
Mean Square Error: 4.232088529212146
Root Mean Square Error: 2.057204056289056
R^2: 0.801686871012805


In [221]:
param_grid_xgb = {
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.001],
    'n_estimators' : [300, 500, 700], 
    'eval_metric' : ['rmse']
    }

grid_search_xgb = HalvingGridSearchCV(
    regressor_2, 
    param_grid_xgb, 
    scoring='r2',
    cv=5, 
    n_jobs=-1,
)

grid_search_xgb.fit(X_train_xgb_2, y_train_xgb_2)

print("Best set of hyperparameters: ", grid_search_xgb.best_params_)
print("Best score: ", grid_search_xgb.best_score_)

Best set of hyperparameters:  {'eval_metric': 'rmse', 'learning_rate': 0.01, 'max_depth': 10, 'n_estimators': 500}
Best score:  0.9534739037687222


## Fields without weather and with year

In [68]:
features_3 = ['Crop_33', 'Crop_34', 'Crop_41', 'Crop_45', 'Crop_174', 'Crop_22',
          'Elevation', 'area', 'Organic_M', 'Ca', 'Mg', 'Mn',
          'B', 'Cu', 'Mo', 'Fe', 'Zn', 'S', 'P', 'K', 'Na', 'pH', 'C.E.C', "year"]

### Regressions

#### MLR

In [69]:
X_mlr_3 = yield_zones_1_[features_3]
y_mlr_3 = yield_zones_1_["VRYIELDMAS"]

scaler_mlr_3 = StandardScaler()
X_mlr_3 = scaler.fit_transform(X_mlr_3)

X_train_mlr_3, X_test_mlr_3, y_train_mlr_3, y_test_mlr_3 = train_test_split(X_mlr_3, y_mlr_3, test_size=0.2, random_state=0)

In [71]:
reg_model_mlr_3 = LinearRegression(copy_X=True, fit_intercept=True, n_jobs=15, positive=True)
reg_model_mlr_3 = LinearRegression().fit(X_train_mlr_3, y_train_mlr_3)
print('Intercept: ', reg_model_mlr_3.intercept_)
y_pred_mlr_3 = reg_model_mlr_3.predict(X_test_mlr_3)  
print("Prediction for test set: {}".format(y_pred_mlr_3))

Intercept:  5.354650273920838
Prediction for test set: [ 1.7958188   2.91751427  1.27962771 14.37512394 12.04027675 11.52114781
  1.86581279  0.92790597 11.28100392 12.077114   12.45477905  4.5454118
  2.07990199  1.19565598  0.9438961   2.67239448  1.01749373 13.21819231
  2.05715364 11.9360893   4.44418823  2.05527595  2.91892261  0.57789603
  3.00976873  1.10998629 12.58951421  2.35986924 13.56697642  2.27816631
  2.90562509  4.29983105  2.92916444  2.30288143 13.86858901  3.90852986
  1.36460508  1.86817728  3.32750362  2.92753082 13.7580708   2.17626645
  3.88694259  2.74103978  2.06973303  2.11741466]


In [72]:
mae_mlr_3 = metrics.mean_absolute_error(y_test_mlr_3, y_pred_mlr_3)
mse_mlr_3 = metrics.mean_squared_error(y_test_mlr_3, y_pred_mlr_3)
rmse_mlr_3 = np.sqrt(metrics.mean_squared_error(y_test_mlr_3, y_pred_mlr_3))
r_2_mlr_3 = r2_score(y_test_mlr_3, y_pred_mlr_3)

print('Mean Absolute Error:', mae_mlr_3)
print('Mean Square Error:', mse_mlr_3)
print('Root Mean Square Error:', rmse_mlr_3)
print('R^2:', r_2_mlr_3)

Mean Absolute Error: 0.8342939386711146
Mean Square Error: 1.213229219696816
Root Mean Square Error: 1.1014668491138606
R^2: 0.9431488067709304


#### Lasso

In [73]:
X_lasso_3 = yield_zones_1_[features_3]
y_lasso_3 = yield_zones_1_["VRYIELDMAS"]

scaler_lasso_3 = StandardScaler()
X_lasso_3 = scaler_lasso_3.fit_transform(X_lasso_3)

X_train_lasso_3, X_test_lasso_3, y_train_lasso_3, y_test_lasso_3 = train_test_split(X_lasso_3, y_lasso_3, test_size=0.2, random_state=0)

In [74]:
lasso_model_3 = Lasso(alpha=0.1)
lasso_model_3.fit(X_train_lasso_3, y_train_lasso_3)

Lasso(alpha=0.1)

In [75]:
y_pred_lasso_3 = lasso_model_3.predict(X_test_lasso_3)

mae_lasso_3 = metrics.mean_absolute_error(y_test_lasso_3, y_pred_lasso_3)
mse_lasso_3 = metrics.mean_squared_error(y_test_lasso_3, y_pred_lasso_3)
rmse_lasso_3 = np.sqrt(metrics.mean_squared_error(y_test_lasso_3, y_pred_lasso_3))
r_2_lasso_3 = r2_score(y_test_lasso_3, y_pred_lasso_3)

print('Mean Absolute Error:', mae_lasso_3)
print('Mean Square Error:', mse_lasso_3)
print('Root Mean Square Error:', rmse_lasso_3)
print('R^2:', r_2_lasso_3)

Mean Absolute Error: 0.7891005520881244
Mean Square Error: 1.1504230597229925
Root Mean Square Error: 1.0725777639514034
R^2: 0.9460918657400673


#### Ridge

In [77]:
X_ridge_3 = yield_zones_1_[features_3]
y_ridge_3 = yield_zones_1_["VRYIELDMAS"]

scaler_ridge_3 = StandardScaler()
X_ridge_3 = scaler_ridge_3.fit_transform(X_ridge_3)

X_train_ridge_3, X_test_ridge_3, y_train_ridge_3, y_test_ridge_3 = train_test_split(X_ridge_3, y_ridge_3, test_size=0.2, random_state=0)

In [78]:
ridge_model_3 = Ridge(alpha=1)
ridge_model_3.fit(X_train_ridge_3, y_train_ridge_3)

Ridge(alpha=1)

In [79]:
y_pred_ridge_3 = ridge_model_3.predict(X_test_ridge_3)

mae_ridge_3 = metrics.mean_absolute_error(y_test_ridge_3, y_pred_ridge_3)
mse_ridge_3 = metrics.mean_squared_error(y_test_ridge_3, y_pred_ridge_3)
rmse_ridge_3 = np.sqrt(metrics.mean_squared_error(y_test_ridge_3, y_pred_ridge_3))
r_2_ridge_3 = r2_score(y_test_ridge_3, y_pred_ridge_3)

print('Mean Absolute Error:', mae_ridge_3)
print('Mean Square Error:', mse_ridge_3)
print('Root Mean Square Error:', rmse_ridge_3)
print('R^2:', r_2_ridge_3)

Mean Absolute Error: 0.7827644916042655
Mean Square Error: 1.0594198856432977
Root Mean Square Error: 1.0292812471056187
R^2: 0.9503562198704074


#### Polynomial

In [80]:
X_pol_3 = yield_zones_1_[features_3]
y_pol_3 = yield_zones_1_["VRYIELDMAS"]

scaler_pol_3 = StandardScaler()
X_pol_3 = scaler_pol_3.fit_transform(X_pol_3)

X_train_pol_3, X_test_pol_3, y_train_pol_3, y_test_pol_3 = train_test_split(X_pol_3, y_pol_3, test_size=0.2, random_state=0)

In [81]:
lin_pol_3 = LinearRegression()
lin_pol_3.fit(X_train_pol_3, y_train_pol_3)

poly_3 = PolynomialFeatures(degree=2)
X_poly_3 = poly_3.fit_transform(X_train_pol_3)
X_poly_test_3 = poly_3.fit_transform(X_test_pol_3)

poly_3.fit(X_poly_3, y_train_pol_3)
lin2_pol_3 = LinearRegression()
lin2_pol_3.fit(X_poly_3, y_train_pol_3)

LinearRegression()

In [82]:
y_pred_pol_3 = lin2_pol_3.predict(X_poly_test_3)

mae_pol_3 = metrics.mean_absolute_error(y_test_pol_3, y_pred_pol_3)
mse_pol_3 = metrics.mean_squared_error(y_test_pol_3, y_pred_pol_3)
rmse_pol_3 = np.sqrt(metrics.mean_squared_error(y_test_pol_3, y_pred_pol_3))
r_2_pol_3 = r2_score(y_test_pol_3, y_pred_pol_3)

print('Mean Absolute Error:', mae_pol_3)
print('Mean Square Error:', mse_pol_3)
print('Root Mean Square Error:', rmse_pol_3)
print('R^2:', r_2_pol_3)

Mean Absolute Error: 36.688822322893884
Mean Square Error: 19129.54072091472
Root Mean Square Error: 138.30958289617794
R^2: -895.3987993792726


In [83]:
mae_ = []
mse_ = []
rmse_ = []
r2_ = []

degrees = [2, 3, 4]

for degree in degrees:
    pipeline = Pipeline([('poly_features', PolynomialFeatures(degree=degree)), ('model', LinearRegression())])
    pipeline.fit(X_train_pol_3, y_train_pol_3)

    y_pred_test_ = pipeline.predict(X_test_pol_3)

    mae_.append(metrics.mean_absolute_error(y_test_pol_3, y_pred_pol_3))
    mse_.append(metrics.mean_squared_error(y_test_pol_3, y_pred_pol_3))
    rmse_.append(np.sqrt(metrics.mean_squared_error(y_test_pol_3, y_pred_pol_3)))
    r2_.append(r2_score(y_test_pol_3, y_pred_test_))

print(degrees,'\n')
print("MAE: ", mae_,'\n')
print("MSE: ", mse_, "\n")
print("RMSE: ", rmse_, "\n")
print("R2: ", r2_, "\n")

[2, 3, 4] 

MAE:  [36.688822322893884, 36.688822322893884, 36.688822322893884] 

MSE:  [19129.54072091472, 19129.54072091472, 19129.54072091472] 

RMSE:  [138.30958289617794, 138.30958289617794, 138.30958289617794] 

R2:  [-895.3987993792726, 0.9314680559247894, 0.9337677652003044] 



#### DT

In [84]:
X_dt_3 = yield_zones_1_[features_3]
y_dt_3 = yield_zones_1_["VRYIELDMAS"]

scaler_dt_3 = StandardScaler()
X_dt_3 = scaler_dt_3.fit_transform(X_dt_3)

X_train_dt_3, X_test_dt_3, y_train_dt_3, y_test_dt_3 = train_test_split(X_dt_3, y_dt_3, test_size=0.2, random_state=0)

In [85]:
regressor_dt_3 = DecisionTreeRegressor(max_depth=10, max_leaf_nodes=16, random_state=42)
regressor_dt_3.fit(X_train_dt_3, y_train_dt_3)

DecisionTreeRegressor(max_depth=10, max_leaf_nodes=16, random_state=42)

In [86]:
y_pred_dt_3 = regressor_dt_3.predict(X_test_dt_3)

mae_dt_3 = metrics.mean_absolute_error(y_test_dt_3, y_pred_dt_3)
mse_dt_3 = metrics.mean_squared_error(y_test_dt_3, y_pred_dt_3)
rmse_dt_3 = np.sqrt(metrics.mean_squared_error(y_test_dt_3, y_pred_dt_3))
r_2_dt_3 = r2_score(y_test_dt_3, y_pred_dt_3)

print('Mean Absolute Error:', mae_dt_3)
print('Mean Square Error:', mse_dt_3)
print('Root Mean Square Error:', rmse_dt_3)
print('R^2:', r_2_dt_3)

Mean Absolute Error: 0.47071301400234644
Mean Square Error: 0.5589428124554049
Root Mean Square Error: 0.7476247805252344
R^2: 0.9738082752055356


#### RF

In [87]:
X_rf_3 = yield_zones_1_[features_3]
y_rf_3 = yield_zones_1_["VRYIELDMAS"]

scaler_rf_3 = StandardScaler()
X_rf_3 = scaler_rf_3.fit_transform(X_rf_3)

X_train_rf_3, X_test_rf_3, y_train_rf_3, y_test_rf_3 = train_test_split(X_rf_3, y_rf_3, test_size=0.2, random_state=0)

In [88]:
regr_rf_3 = RandomForestRegressor(max_depth=10, max_leaf_nodes=32, n_estimators=100)
regr_rf_3.fit(X_train_rf_3, y_train_rf_3)

RandomForestRegressor(max_depth=10, max_leaf_nodes=32)

In [89]:
y_pred_rf_3 = regr_rf_3.predict(X_test_rf_3)

mae_rf_3 = metrics.mean_absolute_error(y_test_rf_3, y_pred_rf_3)
mse_rf_3 = metrics.mean_squared_error(y_test_rf_3, y_pred_rf_3)
rmse_rf_3 = np.sqrt(metrics.mean_squared_error(y_test_rf_3, y_pred_rf_3))
r_2_rf_3 = r2_score(y_test_rf_3, y_pred_rf_3)

print('Mean Absolute Error:', mae_rf_3)
print('Mean Square Error:', mse_rf_3)
print('Root Mean Square Error:', rmse_rf_3)
print('R^2:', r_2_rf_3)

Mean Absolute Error: 0.4343113091760555
Mean Square Error: 0.4774597612893671
Root Mean Square Error: 0.6909846317316812
R^2: 0.9776265220887521


#### KNN

In [90]:
X_knn_3 = yield_zones_1_[features_3]
y_knn_3 = yield_zones_1_["VRYIELDMAS"]

scaler_knn_3 = StandardScaler()
X_knn_3 = scaler_knn_3.fit_transform(X_knn_3)

X_train_knn_3, X_test_knn_3, y_train_knn_3, y_test_knn_3 = train_test_split(X_knn_3, y_knn_3, test_size=0.2, random_state=0)

In [91]:
knn_reg_3 = KNeighborsRegressor(n_neighbors=3)
knn_reg_3.fit(X_train_knn_3, y_train_knn_3)

KNeighborsRegressor(n_neighbors=3)

In [92]:
os.environ['OMP_NUM_THREADS'] = '1'
y_pred_knn_3 = knn_reg_3.predict(X_test_knn_3)

mae_knn_3 = metrics.mean_absolute_error(y_test_knn_3, y_pred_knn_3)
mse_knn_3 = metrics.mean_squared_error(y_test_knn_3, y_pred_knn_3)
rmse_knn_3 = np.sqrt(metrics.mean_squared_error(y_test_knn_3, y_pred_knn_3))
r_2_knn_3 = r2_score(y_test_knn_3, y_pred_knn_3)

print('Mean Absolute Error:', mae_knn_3)
print('Mean Square Error:', mse_knn_3)
print('Root Mean Square Error:', rmse_knn_3)
print('R^2:', r_2_knn_3)

Mean Absolute Error: 1.6955869560768309
Mean Square Error: 8.332474809733824
Root Mean Square Error: 2.8866026414686563
R^2: 0.6095452303704718


#### Elastic Net

In [95]:
X_en_3 = yield_zones_1_[features_3]
y_en_3 = yield_zones_1_["VRYIELDMAS"]

scaler_en_3 = StandardScaler()
X_en_3 = scaler.fit_transform(X_en_3)

X_train_en_3, X_test_en_3, y_train_en_3, y_test_en_3 = train_test_split(X_en_3, y_en_3, test_size=0.2, random_state=0)

In [96]:
regr_en_3 = ElasticNet(alpha=1, tol=1e-4)
regr_en_3.fit(X_train_en_3, y_train_en_3)

ElasticNet(alpha=1)

In [97]:
y_pred_en_3 = regr_en_3.predict(X_test_en_3)

mae_en_3 = metrics.mean_absolute_error(y_test_en_3, y_pred_en_3)
mse_en_3 = metrics.mean_squared_error(y_test_en_3, y_pred_en_3)
rmse_en_3 = np.sqrt(metrics.mean_squared_error(y_test_en_3, y_pred_en_3))
r_2_en_3 = r2_score(y_test_en_3, y_pred_en_3)

print('Mean Absolute Error:', mae_en_3)
print('Mean Square Error:', mse_en_3)
print('Root Mean Square Error:', rmse_en_3)
print('R^2:', r_2_en_3)

Mean Absolute Error: 1.634539036331587
Mean Square Error: 4.174556131549905
Root Mean Square Error: 2.0431730547239275
R^2: 0.8043828046445758


### Gradient Boosting

#### LightGBM

In [104]:
X_gbm_3 = yield_zones_1_[features_3]
y_gbm_3 = yield_zones_1_["VRYIELDMAS"]

scaler_gbm_3 = StandardScaler()
X_gbm_3 = scaler_gbm_3.fit_transform(X_gbm_3)

X_train_gbm_3, X_test_gbm_3, y_train_gbm_3, y_test_gbm_3 = train_test_split(X_gbm_3, y_gbm_3, test_size=0.2, random_state=0)

In [105]:
model_gbm_3 = LGBMRegressor(learning_rate=0.1, max_depth=6, n_estimators=100, num_leaves=64)
model_gbm_3.fit(X_train_gbm_3, y_train_gbm_3)

c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor(max_depth=6, num_leaves=64)

In [106]:
y_pred_gbm_3 = model_gbm_3.predict(X_test_gbm_3)

mae_gbm_3 = metrics.mean_absolute_error(y_test_gbm_3, y_pred_gbm_3)
mse_gbm_3 = metrics.mean_squared_error(y_test_gbm_3, y_pred_gbm_3)
rmse_gbm_3 = np.sqrt(metrics.mean_squared_error(y_test_gbm_3, y_pred_gbm_3))
r_2_gbm_3 = r2_score(y_test_gbm_3, y_pred_gbm_3)

print('Mean Absolute Error:', mae_gbm_3)
print('Mean Square Error:', mse_gbm_3)
print('Root Mean Square Error:', rmse_gbm_3)
print('R^2:', r_2_gbm_3)

Mean Absolute Error: 0.3410606771058329
Mean Square Error: 0.27429033415279797
Root Mean Square Error: 0.5237273471500203
R^2: 0.9871469195312625


c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


#### XGBoost

In [101]:
X_xgb_3 = yield_zones_1_[features_3]
y_xgb_3 = yield_zones_1_["VRYIELDMAS"]

scaler_xgb_3 = StandardScaler()
X_xgb_3 = scaler_xgb_3.fit_transform(X_xgb_3)

X_train_xgb_3, X_test_xgb_3, y_train_xgb_3, y_test_xgb_3 = train_test_split(X_xgb_3, y_xgb_3, test_size=0.2, random_state=0)

In [102]:
regressor_3 = xgb.XGBRegressor(learning_rate = 0.01,
                           n_estimators  = 500,
                           max_depth     = 10,
                           eval_metric   = 'rmse')

regressor_3.fit(X_train_xgb_3, y_train_xgb_3)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [103]:
y_pred_xgb_3 = regressor_3.predict(X_test_xgb_3)

mae_xgb_3 = metrics.mean_absolute_error(y_test_xgb_3, y_pred_xgb_3)
mse_xgb_3 = metrics.mean_squared_error(y_test_xgb_3, y_pred_xgb_3)
rmse_xgb_3 = np.sqrt(metrics.mean_squared_error(y_test_xgb_3, y_pred_xgb_3))
r_2_xgb_3 = r2_score(y_test_xgb_3, y_pred_xgb_3)

print('Mean Absolute Error:', mae_xgb_3)
print('Mean Square Error:', mse_xgb_3)
print('Root Mean Square Error:', rmse_xgb_3)
print('R^2:', r_2_xgb_3)

Mean Absolute Error: 0.411225549310173
Mean Square Error: 0.3896776761603446
Root Mean Square Error: 0.6242416808899776
R^2: 0.9817399379236986
